In [ ]:
# Install dependencies
!pip install -q sentence-transformers

In [ ]:
from google.colab import drive
drive.mount('/content/drive', force_remount=True)


Mounted at /content/drive


In [ ]:
import os

base = '/content/drive/MyDrive/GenAI_project/Data'

for root, dirs, files in os.walk(base):
    level = root.replace(base, '').count(os.sep)
    indent = '  ' * level
    print(f"{indent}{os.path.basename(root)}/")
    for f in files:
        print(f"{'  ' * (level+1)}{f}")

Data/
  lcd_docs.zip
  ncd_docs.zip
  lcd_docs_v2.zip
  lcd_metadata.json
  .DS_Store
  all_chunks.json
  lcd_docs_v2/
    lcd_metadata.json
    .DS_Store
    lcd_docs_v2/
      Electroretinography_ERG_37371.txt
      Immune_Globulin_35093.txt
      Respiratory_Therapy_Respiratory_Care_34430.txt
      Nerve_Blockade_for_Treatment_of_Chronic_Pain_and_Neuropathy_35457.txt
      Bariatric_Surgical_Management_of_Morbid_Obesity_35022.txt
      Outpatient_Sleep_Studies_35050.txt
      Respiratory_Assist_Devices_33800.txt
      Cardiac_Catheterization_and_Coronary_Angiography_33557.txt
      B-type_Natriuretic_Peptide_BNP_Testing_35526.txt
      Ophthalmology_Extended_Ophthalmoscopy_and_Fundus_Photography_33467.txt
      Mohs_Micrographic_Surgery_MMS_34961.txt
      Knee_Orthoses_33318.txt
      Hospice_-_Liver_Disease_34544.txt
      Scanning_Computerized_Ophthalmic_Diagnostic_Imaging_SCODI_34431.txt
      Peripheral_Venous_Ultrasound_35451.txt
      Non-Invasive_Peripheral_Venous_Vascular_a

In [ ]:
!pip install -q pymupdf

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 24.9/24.9 MB 73.2 MB/s eta 0:00:00


In [ ]:
import os

# Update this path to wherever your files are in Drive
base_path = '/content/drive/MyDrive'  # change if in a subfolder


In [ ]:
import os
import json
import re

# ─────────────────────────────────────────
# CONFIGURE THESE PATHS FOR YOUR DRIVE
# ─────────────────────────────────────────
NCD_FOLDER = '/content/drive/MyDrive/GenAI_project/Data/ncd_docs'
LCD_FOLDER = '/content/drive/MyDrive/GenAI_project/Data/lcd_docs_v2/lcd_docs_v2'

OUTPUT_FILE = '/content/drive/MyDrive/GenAI_project/Data/all_chunks.json'

CHUNK_SIZE  = 400   # words per chunk
OVERLAP     = 50    # word overlap between chunks

# ─────────────────────────────────────────
# CHUNKING FUNCTION
# ─────────────────────────────────────────
def chunk_text(text, chunk_size=CHUNK_SIZE, overlap=OVERLAP):
    words = text.split()
    chunks = []
    i = 0
    while i < len(words):
        chunk = ' '.join(words[i:i+chunk_size])
        chunks.append(chunk)
        i += chunk_size - overlap
        if i + overlap >= len(words):
            break
    return chunks

# ─────────────────────────────────────────
# PARSE NCD METADATA FROM FILE HEADER
# ─────────────────────────────────────────
def parse_ncd_file(filepath, filename):
    with open(filepath, encoding='utf-8') as f:
        content = f.read()

    # Extract title from first line if it starts with TITLE:
    lines = content.split('\n')
    title = filename.replace('_', ' ').replace('.txt', '')
    ncd_id = 'UNKNOWN'

    for line in lines[:10]:
        if line.startswith('TITLE:'):
            title = line.replace('TITLE:', '').strip()
        if line.startswith('NCD_ID:') or line.startswith('SECTION:'):
            ncd_id = line.split(':', 1)[-1].strip()

    text_chunks = chunk_text(content)

    results = []
    for i, chunk in enumerate(text_chunks):
        # Prepend metadata so it's never lost even if chunk is shown in isolation
        tagged = f"[TYPE: NCD | NCD_ID: {ncd_id} | TITLE: {title}]\n\n{chunk}"
        results.append({
            'text':      tagged,
            'raw_text':  chunk,
            'source_id': ncd_id,
            'title':     title,
            'type':      'NCD',
            'states':    ['ALL'],   # NCDs apply nationally
            'filename':  filename,
            'chunk_idx': i
        })
    return results

# ─────────────────────────────────────────
# PARSE LCD METADATA FROM FILE HEADER
# ─────────────────────────────────────────
def parse_lcd_file(filepath, filename):
    with open(filepath, encoding='utf-8') as f:
        content = f.read()

    lines = content.split('\n')
    title      = filename.replace('_', ' ').replace('.txt', '')
    lcd_id     = 'UNKNOWN'
    contractor = 'UNKNOWN'
    states     = ['ALL']

    for line in lines[:15]:
        if line.startswith('TITLE:'):
            title = line.replace('TITLE:', '').strip()
        elif line.startswith('LCD_ID:'):
            lcd_id = line.replace('LCD_ID:', '').strip()
        elif line.startswith('CONTRACTOR:'):
            contractor = line.replace('CONTRACTOR:', '').strip()
        elif line.startswith('STATES:'):
            raw = line.replace('STATES:', '').strip()
            states = [s.strip() for s in raw.split(',') if s.strip()]

    text_chunks = chunk_text(content)

    results = []
    for i, chunk in enumerate(text_chunks):
        tagged = f"[TYPE: LCD | LCD_ID: {lcd_id} | TITLE: {title} | STATES: {', '.join(states)} | CONTRACTOR: {contractor}]\n\n{chunk}"
        results.append({
            'text':       tagged,
            'raw_text':   chunk,
            'source_id':  lcd_id,
            'title':      title,
            'type':       'LCD',
            'states':     states,
            'contractor': contractor,
            'filename':   filename,
            'chunk_idx':  i
        })
    return results

# ─────────────────────────────────────────
# PROCESS ALL FILES
# ─────────────────────────────────────────
all_chunks = []
ncd_count  = 0
lcd_count  = 0
skipped    = 0

# Process NCDs
print("Processing NCD files...")
for fname in sorted(os.listdir(NCD_FOLDER)):
    if not fname.endswith('.txt'):
        continue
    fpath = os.path.join(NCD_FOLDER, fname)
    try:
        chunks = parse_ncd_file(fpath, fname)
        all_chunks.extend(chunks)
        ncd_count += 1
    except Exception as e:
        print(f"  Skipped {fname}: {e}")
        skipped += 1

print(f"  ✓ {ncd_count} NCD files → {sum(1 for c in all_chunks if c['type']=='NCD')} chunks")

# Process LCDs
print("Processing LCD files...")
lcd_start = len(all_chunks)
for fname in sorted(os.listdir(LCD_FOLDER)):
    if not fname.endswith('.txt'):
        continue
    fpath = os.path.join(LCD_FOLDER, fname)
    try:
        chunks = parse_lcd_file(fpath, fname)
        all_chunks.extend(chunks)
        lcd_count += 1
    except Exception as e:
        print(f"  Skipped {fname}: {e}")
        skipped += 1

print(f"  ✓ {lcd_count} LCD files → {sum(1 for c in all_chunks if c['type']=='LCD')} chunks")

# ─────────────────────────────────────────
# SAVE OUTPUT
# ─────────────────────────────────────────
with open(OUTPUT_FILE, 'w', encoding='utf-8') as f:
    json.dump(all_chunks, f, indent=2, ensure_ascii=False)

print(f"\n{'='*50}")
print(f"DONE!")
print(f"  Total chunks:  {len(all_chunks)}")
print(f"  NCD files:     {ncd_count}")
print(f"  LCD files:     {lcd_count}")
print(f"  Skipped:       {skipped}")
print(f"  Output saved:  {OUTPUT_FILE}")
print(f"{'='*50}")

Processing NCD files...


FileNotFoundError: [Errno 2] No such file or directory: '/content/drive/MyDrive/GenAI_project/Data/ncd_docs'

In [ ]:
import json

chunks = json.load(open('/content/drive/MyDrive/GenAI_project/Data/all_chunks.json'))

print(f"Total chunks: {len(chunks)}")
print(f"NCD chunks:   {sum(1 for c in chunks if c['type'] == 'NCD')}")
print(f"LCD chunks:   {sum(1 for c in chunks if c['type'] == 'LCD')}")

# Show one NCD chunk
ncd_sample = next(c for c in chunks if c['type'] == 'NCD')
print("\n--- NCD SAMPLE ---")
print(ncd_sample['text'][:400])

# Show one Texas LCD chunk
tx_sample = next((c for c in chunks if 'TX' in c.get('states', [])), None)
print("\n--- TEXAS LCD SAMPLE ---")
print(tx_sample['text'][:400] if tx_sample else "No TX chunk found")

Total chunks: 3377
NCD chunks:   575
LCD chunks:   2802

--- NCD SAMPLE ---
[TYPE: NCD | NCD_ID: 100.3 | TITLE: 24-Hour Ambulatory Esophageal pH Monitoring]

TITLE: 24-Hour Ambulatory Esophageal pH Monitoring SECTION: 100.3 EFFECTIVE DATE: 1985-06-11 00:00:00 COVERAGE POLICY: Twenty-four hour ambulatory pH monitoring is covered by Medicare for patients who are suspected of having gastric reflux, but only if the patient presents diagnostic problems associated with atypical

--- TEXAS LCD SAMPLE ---
[TYPE: LCD | LCD_ID: 37792 | TITLE: 4Kscore Test Algorithm | STATES: AL, AK, AZ, AR, CA, CO, CT, DE, DC, FL, GA, HI, ID, IL, IN, IA, KS, KY, LA, ME, MD, MA, MI, MN, MS, MO, MT, NE, NV, NH, NJ, NM, NY, NC, ND, OH, OK, OR, PA, PR, RI, SC, SD, TN, TX, UT, VT, VA, VI, WA, WV, WI, WY | CONTRACTOR: NATIONAL]

TITLE: 4Kscore Test Algorithm LCD_ID: 37792 TYPE: Local Coverage Determination (LCD) CONTRACTOR


In [ ]:
import fitz  # pymupdf
import os
import json
import re

# ─── PATHS ───────────────────────────────────────────────────────────────────
BENEFIT_FOLDER   = '/content/drive/MyDrive/GenAI_project/Data/Medicare Benefit Policy Manual'
CLAIMS_FOLDER    = '/content/drive/MyDrive/GenAI_project/Data/Medicare Claims Processing'
EXISTING_CHUNKS  = '/content/drive/MyDrive/GenAI_project/Data/all_chunks.json'
OUTPUT_FILE      = '/content/drive/MyDrive/GenAI_project/Data/all_chunks.json'

CHUNK_SIZE = 400
OVERLAP    = 50

# ─── CHUNKING ────────────────────────────────────────────────────────────────
def chunk_text(text, chunk_size=CHUNK_SIZE, overlap=OVERLAP):
    words = text.split()
    chunks = []
    i = 0
    while i < len(words):
        chunk = ' '.join(words[i:i + chunk_size])
        chunks.append(chunk)
        i += chunk_size - overlap
        if i + overlap >= len(words):
            break
    return chunks

# ─── PDF TEXT EXTRACTION ─────────────────────────────────────────────────────
def extract_pdf_text(filepath):
    doc = fitz.open(filepath)
    full_text = []
    for page in doc:
        text = page.get_text("text")
        if text.strip():
            full_text.append(text.strip())
    doc.close()
    return '\n'.join(full_text)

# ─── PROCESS ONE PDF FOLDER ──────────────────────────────────────────────────
def process_pdf_folder(folder_path, doc_type, manual_name):
    results = []
    files = sorted([f for f in os.listdir(folder_path)
                    if f.endswith('.pdf') and not f.startswith('.')])

    print(f"\nProcessing {len(files)} PDFs from {manual_name}...")

    for fname in files:
        fpath = os.path.join(folder_path, fname)
        chapter_title = fname.replace('.pdf', '').strip()

        try:
            raw_text = extract_pdf_text(fpath)

            if len(raw_text.split()) < 50:
                print(f"  ⚠ Skipped (too short): {fname}")
                continue

            text_chunks = chunk_text(raw_text)

            for i, chunk in enumerate(text_chunks):
                header = (f"[TYPE: {doc_type} | MANUAL: {manual_name} "
                         f"| CHAPTER: {chapter_title} | STATES: ALL]")
                tagged = f"{header}\n\n{chunk}"

                results.append({
                    'text':          tagged,
                    'raw_text':      chunk,
                    'source_id':     chapter_title,
                    'title':         chapter_title,
                    'type':          doc_type,
                    'manual':        manual_name,
                    'states':        ['ALL'],
                    'filename':      fname,
                    'chunk_idx':     i
                })

            print(f"  ✓ {fname[:60]} → {len(text_chunks)} chunks")

        except Exception as e:
            print(f"  ✗ Error on {fname}: {e}")

    return results

# ─── MAIN ────────────────────────────────────────────────────────────────────

# Load existing chunks
print("Loading existing chunks...")
with open(EXISTING_CHUNKS, 'r') as f:
    existing_chunks = json.load(f)
print(f"  Existing chunks: {len(existing_chunks)}")

# Process both manuals
benefit_chunks = process_pdf_folder(
    BENEFIT_FOLDER,
    doc_type='BENEFIT_POLICY',
    manual_name='Medicare Benefit Policy Manual'
)

claims_chunks = process_pdf_folder(
    CLAIMS_FOLDER,
    doc_type='CLAIMS_PROCESSING',
    manual_name='Medicare Claims Processing Manual'
)

# Merge everything
all_chunks = existing_chunks + benefit_chunks + claims_chunks

# Save
with open(OUTPUT_FILE, 'w', encoding='utf-8') as f:
    json.dump(all_chunks, f, indent=2, ensure_ascii=False)

# ─── SUMMARY ─────────────────────────────────────────────────────────────────
print(f"\n{'='*55}")
print(f"DONE!")
print(f"  Previous chunks:         {len(existing_chunks):,}")
print(f"  Benefit Policy chunks:   {len(benefit_chunks):,}")
print(f"  Claims Processing chunks:{len(claims_chunks):,}")
print(f"  TOTAL chunks:            {len(all_chunks):,}")
print(f"  Saved to: {OUTPUT_FILE}")
print(f"{'='*55}")

Loading existing chunks...
  Existing chunks: 3377

Processing 17 PDFs from Medicare Benefit Policy Manual...
  ✓ Chapter 1 - Inpatient Hospital Services Covered Under .pdf → 62 chunks
  ✓ Chapter 10 - Ambulance Services .pdf → 33 chunks
  ✓ Chapter 11 - End Stage Renal Disease (ESRD).pdf → 89 chunks
  ✓ Chapter 12 - Comprehensive Outpatient Rehabilitation Facilit → 20 chunks
  ✓ Chapter 13 - Rural Health Clinic (RHC) and Federally Qualifi → 73 chunks
  ✓ Chapter 14 - Medical Devices.pdf → 7 chunks
  ✓ Chapter 15 – Covered Medical and Other Health Services.pdf → 349 chunks
  ✓ Chapter 16 - General Exclusions From Coverage.pdf → 41 chunks
  ✓ Chapter 17 - Opioid Treatment Programs (OTPs).pdf → 19 chunks
  ✓ Chapter 2 - Inpatient Psychiatric Hospital Services .pdf → 17 chunks
  ✓ Chapter 3 - Duration of Covered Inpatient Services.pdf → 5 chunks
  ✓ Chapter 4 - Inpatient Psychiatric Benefit Days Reduction and → 6 chunks
  ✓ Chapter 5 - Lifetime Reserve Days.pdf → 16 chunks
  ✓ Chapter 6 -